# Full Re‑ID Pipeline – Transparent Run
This notebook runs the complete pipeline **step‑by‑step** and prints every important decision:  
detections → track counting → crop quality filtering → online global linking → offline refinement (split/merge) → final visualisation.

**First, set your paths below** and then execute all cells in order.

In [15]:
# ========== EDIT THESE TWO STRINGS ==========
INPUT_VIDEO = "../assets/store_cam3.mp4"          # <-- your video file
OUTPUT_DIR  = "./outputs/test3"        # <-- where to save everything
# ============================================

In [16]:
# Ensure CUDA device visibility is set before importing torch
import os

cuda_visible = os.environ.get("CUDA_VISIBLE_DEVICES")
if cuda_visible in (None, "", "-1"):
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("NVIDIA_VISIBLE_DEVICES:", os.environ.get("NVIDIA_VISIBLE_DEVICES"))

CUDA_VISIBLE_DEVICES: 0
NVIDIA_VISIBLE_DEVICES: None


In [ ]:
import sys, time, warnings, shutil, os, re
from pathlib import Path
from collections import defaultdict
import numpy as np
import cv2
import torch
from tqdm.notebook import tqdm

# Project root
PROJECT_ROOT = Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config

def print_config_summary():
    print("=== Config values in use ===")
    print(f"OSNET_MODEL: {config.OSNET_MODEL}")
    print("Step2 (online) quality gates:")
    print(f"  MIN_REID_WIDTH={config.MIN_REID_WIDTH}, MIN_REID_HEIGHT={config.MIN_REID_HEIGHT}")
    print(f"  MIN_REID_AREA_RATIO={config.MIN_REID_AREA_RATIO}, MIN_REID_SHARPNESS={config.MIN_REID_SHARPNESS}")
    print("Save gates (crop saving):")
    print(f"  MIN_WIDTH={config.MIN_WIDTH}, MIN_HEIGHT={config.MIN_HEIGHT}")
    print(f"  MIN_AREA_RATIO={config.MIN_AREA_RATIO}, MIN_SHARPNESS={config.MIN_SHARPNESS}")
    print("Stage A (offline quality gates):")
    print(f"  REFINE_MIN_REID_AREA_RATIO={config.REFINE_MIN_REID_AREA_RATIO}")
    print(f"  REFINE_MIN_REID_SHARPNESS={config.REFINE_MIN_REID_SHARPNESS}")
    print(f"  REFINE_MAX_ASPECT_RATIO={config.REFINE_MAX_ASPECT_RATIO}")
    print("Stage B (split thresholds):")
    print(f"  MIN_TRACK_SIZE={config.MIN_TRACK_SIZE}")
    print(f"  SPLIT_REID_THRESHOLD={config.SPLIT_REID_THRESHOLD}")
    print(f"  SPLIT_COLOR_THRESHOLD={config.SPLIT_COLOR_THRESHOLD}")
    print("Stage C (merge thresholds):")
    print(f"  MERGE_REID_THRESHOLD={config.MERGE_REID_THRESHOLD}")
    print(f"  MERGE_COLOR_THRESHOLD={config.MERGE_COLOR_THRESHOLD}")
    print(f"  MERGE_COLOR_WEIGHT={config.MERGE_COLOR_WEIGHT}")
    print(f"  COHERENCE_MIN={config.COHERENCE_MIN}")
    print(f"  MAX_SAME_FRAME_OVERLAP={config.MAX_SAME_FRAME_OVERLAP}")
    print(f"  MIN_FOLDER_IMAGES_FINAL={config.MIN_FOLDER_IMAGES_FINAL}")
    print("Large-folder relaxations:")
    print(f"  LARGE_FOLDER_SIZE={config.LARGE_FOLDER_SIZE}")
    print(f"  LARGE_MERGE_REID_THRESHOLD={config.LARGE_MERGE_REID_THRESHOLD}")
    print(f"  LARGE_MERGE_COLOR_THRESHOLD={config.LARGE_MERGE_COLOR_THRESHOLD}")

print_config_summary()

from modules.detector_tracker import YOLOTracker
from modules.reid_embedder import ReIDEmbedder
from modules.global_id_manager import GlobalIDManager

# Notebook-specific output folders
INPUT_VIDEO_PATH = Path(INPUT_VIDEO)
OUTPUT_BASE = Path(OUTPUT_DIR)
STEP2_CROPS = OUTPUT_BASE / "step2_crops"
REFINED_DIR  = OUTPUT_BASE / "refined"
FINAL_VIDEO  = OUTPUT_BASE / "final_visualization.mp4"

# Create output directories
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
print(f"Input video  : {INPUT_VIDEO_PATH}")
print(f"Output folder: {OUTPUT_BASE}")

Input video  : ../assets/store_cam3.mp4
Output folder: outputs/test3


In [18]:
# Print virtual environment info
venv_env = os.environ.get('VIRTUAL_ENV') or os.environ.get('CONDA_PREFIX')
sys_exec = sys.executable
sys_pref = sys.prefix
base_pref = getattr(sys, "base_prefix", None)
project_venv = PROJECT_ROOT / ".venv"

print("sys.executable:", sys_exec)
print("sys.prefix   :", sys_pref)
print("sys.base_prefix:", base_pref)
print("VIRTUAL_ENV / CONDA_PREFIX:", venv_env)
print("PROJECT_ROOT/.venv exists?:", project_venv.exists(), "->", str(project_venv.resolve()) if project_venv.exists() else str(project_venv))

# Is the current python executable located inside PROJECT_ROOT/.venv?
try:
    in_project_venv = project_venv.resolve() in Path(sys_exec).resolve().parents
except Exception:
    in_project_venv = False
print("sys.executable inside PROJECT_ROOT/.venv?:", in_project_venv)

sys.executable: /home/zeno/Documents/esi/the-project/.venv/bin/python
sys.prefix   : /home/zeno/Documents/esi/the-project/.venv
sys.base_prefix: /usr
VIRTUAL_ENV / CONDA_PREFIX: /home/zeno/Documents/esi/the-project/.venv
PROJECT_ROOT/.venv exists?: False -> /home/zeno/Documents/esi/the-project/reid_system/.venv
sys.executable inside PROJECT_ROOT/.venv?: False


In [19]:
# Check GPU availability
print("PyTorch version:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("CUDA device count:", torch.cuda.device_count())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not available – everything will run on CPU.")
    print("Tip: if you changed CUDA_VISIBLE_DEVICES, restart the kernel.")

PyTorch version: 2.2.2+cu121
torch.version.cuda: 12.1
CUDA device count: 1
CUDA available: True
Device: NVIDIA GeForce GTX 1050


In [20]:
# Load video info
cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))
if not cap.isOpened():
    raise IOError(f"Cannot open video: {INPUT_VIDEO_PATH}")
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"Video: {INPUT_VIDEO_PATH.name}")
print(f"Frames: {total_frames}, FPS: {fps:.2f}, Resolution: {width}x{height}")
print(f"Duration: {total_frames/fps:.1f}s")

Video: store_cam3.mp4
Frames: 379, FPS: 24.00, Resolution: 3840x2160
Duration: 15.8s


## Step 1 – YOLO + ByteTrack (Local Tracking)
We only run the tracker to see how many local tracks appear. No ReID yet.

In [21]:
tracker = YOLOTracker()
cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))

local_tracks_all = defaultdict(list)   # local_id -> list of frame indices
frame_idx = 0
with tqdm(total=total_frames, desc="Tracking") as pbar:
    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_idx += 1
        dets = tracker.track(frame, persist=True)
        for d in dets:
            local_tracks_all[d['local_id']].append(frame_idx)
        pbar.update(1)
cap.release()

print(f"\nTotal frames processed: {frame_idx}")
print(f"Distinct local track IDs: {len(local_tracks_all)}")
# Print some track lengths
track_lens = sorted([(lid, len(fr)) for lid, fr in local_tracks_all.items()], key=lambda x: x[1], reverse=True)
print("Top 10 longest local tracks:")
for lid, cnt in track_lens[:10]:
    print(f"  L{lid}: {cnt} frames")

Tracking:   0%|          | 0/379 [00:00<?, ?it/s]


Total frames processed: 379
Distinct local track IDs: 59
Top 10 longest local tracks:
  L9: 374 frames
  L45: 333 frames
  L30: 329 frames
  L1: 310 frames
  L97: 257 frames
  L2: 254 frames
  L10: 253 frames
  L114: 229 frames
  L119: 221 frames
  L121: 219 frames


## Step 2 – Online Global ID Linking (with ReID and Colour)
Now we run the full online pipeline.  
**Quality gates** are applied: only crops passing `MIN_REID_AREA_RATIO` and `MIN_REID_SHARPNESS` are used for ReID.  
The `GlobalIDManager` decides whether a detection gets a new global ID or is linked to an existing one.  
We save the crops under `person_XXX` folders.

In [22]:
STEP2_CROPS.mkdir(parents=True, exist_ok=True)

tracker2 = YOLOTracker()
embedder = ReIDEmbedder(model_path=config.OSNET_MODEL)
manager = GlobalIDManager()
manager.reset_for_video(fps)

cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))
frame_idx = 0
total_saved = 0
saved_per_global = defaultdict(int)
filtered_reid = 0  # count of crops that failed ReID quality gates

print("Processing video with online global linking ...")
pbar = tqdm(total=total_frames, desc="Step2 linking")

while True:
    ret, frame = cap.read()
    if not ret: break
    frame_idx += 1
    manager.new_frame()
    dets = tracker2.track(frame, persist=True)
    for det in dets:
        lid = det['local_id']
        crop = det['crop']
        x1,y1,x2,y2 = det['bbox']
        bw, bh = x2-x1, y2-y1
        area_ratio = (bw*bh)/(width*height)
        sharp = cv2.Laplacian(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY), cv2.CV_32F).var()
        reid_ok = (bw >= config.MIN_REID_WIDTH and bh >= config.MIN_REID_HEIGHT and
                   area_ratio >= config.MIN_REID_AREA_RATIO and sharp >= config.MIN_REID_SHARPNESS)
        if reid_ok:
            emb = embedder.extract_embedding(crop)
            color_sig = embedder.extract_color_signature(crop)
            gid, is_new, reason = manager.update(lid, frame_idx, emb, color_sig, (x1,y1,x2,y2), reid_ok=True)
        else:
            filtered_reid += 1
            gid, is_new, reason = manager.update(lid, frame_idx, None, None, (x1,y1,x2,y2), reid_ok=False)
        # Save crop if it passes the standard save quality gates
        save_ok = (bw >= config.MIN_WIDTH and bh >= config.MIN_HEIGHT and
                   area_ratio >= config.MIN_AREA_RATIO and sharp >= config.MIN_SHARPNESS)
        if save_ok:
            person_dir = STEP2_CROPS / f"person_{gid:03d}"
            person_dir.mkdir(parents=True, exist_ok=True)
            ts = frame_idx / fps
            fname = f"frame_{frame_idx:06d}_t{ts:.3f}_l{lid}.jpg"
            cv2.imwrite(str(person_dir / fname), crop,
                        [cv2.IMWRITE_JPEG_QUALITY, config.JPEG_QUALITY])
            total_saved += 1
            saved_per_global[gid] += 1
    manager.cleanup_stale(frame_idx)
    pbar.update(1)

pbar.close()
cap.release()

# Summary
print(f"\n--- Step2 Summary ---")
print(f"Global identities created: {manager.next_id - 1}")
print(f"Total crops saved: {total_saved}")
print(f"Crops discarded by ReID quality gate: {filtered_reid}")
print("Crops per global ID (top 10):")
for gid in sorted(saved_per_global, key=saved_per_global.get, reverse=True)[:10]:
    print(f"  person_{gid:03d}: {saved_per_global[gid]} crops")

ReID embedder loaded (ONNX, CPU). Input size: 128x256
Processing video with online global linking ...


Step2 linking:   0%|          | 0/379 [00:00<?, ?it/s]


--- Step2 Summary ---
Global identities created: 36
Total crops saved: 5587
Crops discarded by ReID quality gate: 55
Crops per global ID (top 10):
  person_009: 374 crops
  person_006: 361 crops
  person_011: 358 crops
  person_001: 351 crops
  person_010: 347 crops
  person_013: 335 crops
  person_014: 333 crops
  person_012: 304 crops
  person_018: 273 crops
  person_002: 254 crops


## Step 3 – Offline Refinement (Pass 2)
This stage:
1. Filters bad crops again (Stage A).
2. Splits folders that contain multiple people (Stage B).
3. Merges folders that belong to the same person (Stage C).

Every decision is printed below.

In [23]:
from scripts.offline_refine_v2 import (
    load_osnet_onnx, is_good_crop, TrackCluster,
    split_folder_tracks, cross_folder_merge, merge_clusters,
    extract_embedding as refine_extract_emb,
    extract_color_signature as refine_extract_color,
    FILENAME_RE as REFINE_RE,
)

# Prepare output
if REFINED_DIR.exists():
    shutil.rmtree(REFINED_DIR)
REFINED_DIR.mkdir()

# Load ReID model
reid_path = Path(config.OSNET_MODEL)
sess, iname = load_osnet_onnx(reid_path)

# We'll use dummy frame dims for area ratio (the other filters are enough)
frame_w, frame_h = 100000, 100000

# Gather folders
folders = sorted([d for d in STEP2_CROPS.iterdir() if d.is_dir() and d.name.startswith("person_")],
                 key=lambda d: d.name)
print(f"Input folders to refine: {len(folders)}")

Input folders to refine: 36


In [ ]:
# ---- Stage A & B: Intra-folder processing with verbose output ----
all_clusters = []

for folder in tqdm(folders, desc="Intra-folder"):
    images = sorted([p for p in folder.iterdir() if p.suffix.lower() in {".jpg",".jpeg",".png"}])
    original_count = len(images)
    if original_count == 0:
        continue

    # Stage A filter
    tracks = defaultdict(lambda: TrackCluster(-1))
    filtered_out = 0
    for img_path in images:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        if not is_good_crop(img, w, h, frame_w, frame_h):
            filtered_out += 1
            continue
        m = REFINE_RE.search(img_path.stem)
        local_id = int(m.group(2)) if m else -1
        frame_no = int(m.group(1)) if m else -1
        emb = refine_extract_emb(img, sess, iname, config.REID_SIZE)
        up, lo = refine_extract_color(img)
        tracks[local_id].add(img_path, emb, up, lo, frame_no)

    print(f"\n{'='*60}")
    print(f"Folder: {folder.name} | original images: {original_count} | kept after quality filter: {original_count - filtered_out}")
    print(f"Number of local tracks: {len(tracks)}")

    # Build prototypes and show track sizes
    for lid, tc in tracks.items():
        tc.build_prototype()
        coh = tc.coherence if tc.coherence is not None else float("nan")
        print(f"  L{lid}: {tc.size()} images, coherence {coh:.3f}")

    print("Split thresholds:")
    print(f"  MIN_TRACK_SIZE={config.MIN_TRACK_SIZE}")
    print(f"  SPLIT_REID_THRESHOLD={config.SPLIT_REID_THRESHOLD}")
    print(f"  SPLIT_COLOR_THRESHOLD={config.SPLIT_COLOR_THRESHOLD}")

    # Show which tracks are considered tiny vs large (tiny are discarded)
    large_ids = [lid for lid, tc in tracks.items() if tc.size() >= config.MIN_TRACK_SIZE]
    tiny_ids = [lid for lid, tc in tracks.items() if tc.size() < config.MIN_TRACK_SIZE]
    if tiny_ids:
        preview = tiny_ids[:20]
        suffix = " ..." if len(tiny_ids) > 20 else ""
        print(f"  Tiny tracks (<{config.MIN_TRACK_SIZE}) discarded: {preview}{suffix}")
    else:
        print(f"  Tiny tracks (<{config.MIN_TRACK_SIZE}) discarded: none")

    # Pairwise best-match summary to explain split decisions
    if len(tracks) > 1:
        track_items = list(tracks.items())
        for lid, tc in track_items:
            best = None
            for other_lid, other in track_items:
                if other_lid == lid:
                    continue
                reid, col, _ = tc.similarity_to(other)
                if best is None or reid + col > best[0]:
                    best = (reid + col, other_lid, reid, col)
            if best is not None:
                _, other_lid, reid, col = best
                passes = (reid >= config.SPLIT_REID_THRESHOLD and col >= config.SPLIT_COLOR_THRESHOLD)
                print(f"  Best match for L{lid} -> L{other_lid}: reid={reid:.3f}, color={col:.3f}, pass={passes}")

    # Stage B – intra-folder splitting (verbose)
    clusters_from_folder = split_folder_tracks(dict(tracks), verbose=True)
    print(f"Result: {len(clusters_from_folder)} cluster(s) after intra-folder split.")
    all_clusters.extend(clusters_from_folder)

print(f"\nTotal clusters before cross-folder merge: {len(all_clusters)}")

Intra-folder:   0%|          | 0/36 [00:00<?, ?it/s]


Folder: person_001 | original images: 351 | kept after quality filter: 351
Number of local tracks: 2
  L1: 310 images, coherence 0.905
  L207: 41 images, coherence 0.959
Result: 2 cluster(s) after intra-folder split.

Folder: person_002 | original images: 254 | kept after quality filter: 254
Number of local tracks: 1
  L2: 254 images, coherence 0.848
Result: 1 cluster(s) after intra-folder split.

Folder: person_003 | original images: 141 | kept after quality filter: 141
Number of local tracks: 1
  L3: 141 images, coherence 0.898
Result: 1 cluster(s) after intra-folder split.

Folder: person_004 | original images: 202 | kept after quality filter: 202
Number of local tracks: 1
  L4: 202 images, coherence 0.904
Result: 1 cluster(s) after intra-folder split.

Folder: person_005 | original images: 193 | kept after quality filter: 193
Number of local tracks: 1
  L5: 193 images, coherence 0.873
Result: 1 cluster(s) after intra-folder split.

Folder: person_006 | original images: 361 | kept 

In [25]:
# ---- Stage C: Cross-folder merging with verbose output ----
final_clusters = cross_folder_merge(all_clusters, verbose=True)
print(f"\nClusters after merging: {len(final_clusters)}")

Cross-folder merging: 100%|██████████| 31/31 [00:00<00:00, 1366.21it/s]

  New person cluster (size 374)
  New person cluster (size 358)
  New person cluster (size 333)
  New person cluster (size 329)
  New person cluster (size 310)
  New person cluster (size 257)
  New person cluster (size 254)
  New person cluster (size 253)
  New person cluster (size 229)
  New person cluster (size 221)
  New person cluster (size 204)
  New person cluster (size 202)
  New person cluster (size 197)
  New person cluster (size 193)
  New person cluster (size 179)
  New person cluster (size 172)
  Merged cluster (size 170) into existing (size=342, score=1.007)
  New person cluster (size 166)
  New person cluster (size 141)
  Merged cluster (size 116) into existing (size=370, score=1.010)
  New person cluster (size 99)
  New person cluster (size 96)
  Merged cluster (size 95) into existing (size=236, score=1.029)
  New person cluster (size 93)
  New person cluster (size 90)
  New person cluster (size 61)
  New person cluster (size 44)
  New person cluster (size 42)
  New pers

In [26]:
# ---- Write final refined folders ----
final_clusters.sort(key=lambda c: c.size(), reverse=True)
out_idx = 0
for cluster in final_clusters:
    if cluster.size() < config.MIN_FOLDER_IMAGES_FINAL:
        print(f"Dropping cluster of size {cluster.size()} (min required: {config.MIN_FOLDER_IMAGES_FINAL})")
        continue
    out_idx += 1
    person_dir = REFINED_DIR / f"person_{out_idx:03d}"
    person_dir.mkdir()
    for src in cluster.paths:
        dest = person_dir / src.name
        if dest.exists():
            i = 1
            while (person_dir / f"{src.stem}_{i}{src.suffix}").exists():
                i += 1
            dest = person_dir / f"{src.stem}_{i}{src.suffix}"
        shutil.copy2(src, dest)

print(f"Final refined persons: {out_idx}")

Final refined persons: 27


## Final Visualisation
We replay the video and draw bounding boxes with the **refined person IDs**.

In [27]:
# Build identity map from refined folders
identity_map = {}
for folder in sorted(REFINED_DIR.iterdir()):
    if not folder.is_dir() or not folder.name.startswith("person_"):
        continue
    pid = int(folder.name.split("_")[1])
    for img_file in folder.iterdir():
        if img_file.suffix.lower() not in {".jpg",".jpeg",".png"}:
            continue
        m = REFINE_RE.search(img_file.stem)
        if m:
            frame_no = int(m.group(1))
            local_id = int(m.group(2))
            identity_map[(frame_no, local_id)] = pid

print(f"Loaded {len(set(identity_map.values()))} distinct refined identities for visualisation.")

Loaded 27 distinct refined identities for visualisation.


In [28]:
# Visualization
cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))
out_video = cv2.VideoWriter(str(FINAL_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
tracker_vis = YOLOTracker()

def id_to_color(id_val):
    hue = ((id_val * 0.61803398875) % 1.0) * 179.0
    hsv = np.array([[[hue, 220, 255]]], dtype=np.uint8)
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0,0]
    return int(bgr[0]), int(bgr[1]), int(bgr[2])

frame_idx = 0
pbar = tqdm(total=total_frames, desc="Visualizing")
while True:
    ret, frame = cap.read()
    if not ret: break
    frame_idx += 1
    dets = tracker_vis.track(frame, persist=True)
    annotated = frame.copy()
    for d in dets:
        x1,y1,x2,y2 = d['bbox']
        lid = d['local_id']
        pid = identity_map.get((frame_idx, lid))
        if pid is not None:
            color = id_to_color(pid)
            label = f"P{pid}"
        else:
            color = (128,128,128)
            label = "?"
        cv2.rectangle(annotated, (x1,y1), (x2,y2), color, 2)
        cv2.putText(annotated, label, (x1, max(y1-10, 20)),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, color, 1, cv2.LINE_AA)
    out_video.write(annotated)
    pbar.update(1)
pbar.close()
cap.release()
out_video.release()
print(f"Visualisation saved to {FINAL_VIDEO}")

Visualizing:   0%|          | 0/379 [00:00<?, ?it/s]

Visualisation saved to outputs/test3/final_visualization.mp4


## Total Pipeline Time
*(Times printed above are per stage; total is displayed after all cells.)* 